In [86]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input 
from tensorflow.keras.layers import SimpleRNN , LSTM , Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [87]:
data = '''The Thing the Time Traveller held quack in his hand was a glittering metallic framework, scarcely larger than a small clock, and very delicately made. There was ivory in it, and some transparent crystalline substance. And now I must be explicit, for this that follows—unless his explanation is to be accepted—is an absolutely unaccountable thing. He took one of the small octagonal tables that were scattered about the room, and set it in front of the fire, with two legs on the hearthrug. On this table he placed the mechanism. Then he drew up a chair, and sat down. The only other object on the table was a small shaded lamp, the bright light of which fell upon the model. There were also perhaps a dozen candles about, two in brass candlesticks upon the mantel and several in sconces, so that the room was brilliantly illuminated. I sat in a low arm-chair nearest the fire, and I drew this forward so as to be almost between the Time Traveller and the fireplace. Filby sat behind him, looking over his shoulder. The Medical Man and the Provincial Mayor watched him in profile from the right, the Psychologist from the left. The Very Young Man stood behind the Psychologist. We were all on the alert. It appears incredible to me that any kind of trick, however subtly conceived and however adroitly done, could have been played upon us under these conditions.
'''

In [88]:
token = Tokenizer()
token.fit_on_texts([data])

#For word index
token.word_index

{'the': 1,
 'and': 2,
 'in': 3,
 'a': 4,
 'was': 5,
 'that': 6,
 'of': 7,
 'on': 8,
 'his': 9,
 'small': 10,
 'it': 11,
 'i': 12,
 'be': 13,
 'this': 14,
 'to': 15,
 'he': 16,
 'were': 17,
 'sat': 18,
 'upon': 19,
 'thing': 20,
 'time': 21,
 'traveller': 22,
 'very': 23,
 'there': 24,
 'about': 25,
 'room': 26,
 'fire': 27,
 'two': 28,
 'table': 29,
 'drew': 30,
 'chair': 31,
 'so': 32,
 'behind': 33,
 'him': 34,
 'man': 35,
 'from': 36,
 'psychologist': 37,
 'however': 38,
 'held': 39,
 'quack': 40,
 'hand': 41,
 'glittering': 42,
 'metallic': 43,
 'framework': 44,
 'scarcely': 45,
 'larger': 46,
 'than': 47,
 'clock': 48,
 'delicately': 49,
 'made': 50,
 'ivory': 51,
 'some': 52,
 'transparent': 53,
 'crystalline': 54,
 'substance': 55,
 'now': 56,
 'must': 57,
 'explicit': 58,
 'for': 59,
 'follows—unless': 60,
 'explanation': 61,
 'is': 62,
 'accepted—is': 63,
 'an': 64,
 'absolutely': 65,
 'unaccountable': 66,
 'took': 67,
 'one': 68,
 'octagonal': 69,
 'tables': 70,
 'scattered':

In [89]:
input_seq = []
# spliting the data into sentences
for sent in data.split('.'):
    # token.texts_to_sequences([sent])
    tokenised_data = token.texts_to_sequences([sent])[0]
    for i in range(1, len(tokenised_data)):
        n_gram = tokenised_data[:i+1]
        input_seq.append(n_gram)

In [90]:
# padding the input sequence
input_padded = pad_sequences(input_seq, padding='pre')
print(input_padded)

[[  0   0   0 ...   0   1  20]
 [  0   0   0 ...   1  20   1]
 [  0   0   0 ...  20   1  21]
 ...
 [  0   0   0 ...  19 141 142]
 [  0   0   0 ... 141 142 143]
 [  0   0  11 ... 142 143 144]]


In [91]:
# Spliting the data into input and output

X = input_padded[:,:-1]
y = input_padded[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2)

In [92]:
X_train.shape, y_train.shape

((180, 27), (180,))

In [93]:
# Converting the output column into categorical data

from tensorflow.keras.utils import to_categorical
y_train_categorical = to_categorical(y_train, num_classes=len(token.word_index)+1)
y_test_categorical = to_categorical(y_test, num_classes=len(token.word_index)+1)

In [94]:
# Architecting the model
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Embedding(input_dim=len(token.word_index)+1, output_dim=10, input_length=X_train.shape[1]),
    LSTM(150),
    Dense(len(token.word_index)+1, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\biswa\Anaconda3PythonLatest2025\envs\venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 27, 10)         │         1,450 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 150)            │        96,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 145)            │        21,895 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119,945 (468.54 KB)

 Trainable params: 119,945 (468.54 KB)

 Non-trainable params: 0 (0.00 B)

In [95]:
model.fit(X_train, y_train_categorical, epochs=100, validation_data=(X_test, y_test_categorical))

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - accuracy: 0.0500 - loss: 4.9746 - val_accuracy: 0.1087 - val_loss: 4.9711
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0833 - loss: 4.9493 - val_accuracy: 0.1087 - val_loss: 4.9391
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0833 - loss: 4.7632 - val_accuracy: 0.1087 - val_loss: 5.1419
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0833 - loss: 4.7097 - val_accuracy: 0.1087 - val_loss: 5.0965
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0833 - loss: 4.6356 - val_accuracy: 0.1087 - val_loss: 5.2217
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0833 - loss: 4.6007 - val_accuracy: 0.1087 - val_loss: 5.3961
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.0833 - loss: 4.5876 - val_accuracy: 0.1087 - val_loss: 5.5101
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0833 - loss: 4.5755 - val_accuracy: 0.1087 - val_loss:

In [97]:
import numpy as np
text = "nigga what"

for i in range(10):
    # tokenize
    token_text = token.texts_to_sequences([text])[0]
    # padding
    pad_text = pad_sequences([token_text], maxlen=X_train.shape[1], padding='pre')
    # predict
    word = token.index_word[np.argmax(model.predict(pad_text))]
    text += " " + word

print(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
nigga what sat sat sat a low chair chair nearest fire fire
